In [ ]:
# ---------------- CELL 1: shared setup (run first) ----------------
# Which prime powers to verify, and the combinatorics shared by the cache
# builder and the main pipeline.

#D_LIST = [5, 7, 8, 9]          # prime powers d >= 5
L = 7 #set limit
D_LIST = [x for x in range(1, L+1) if is_prime_power(x)]
def nd_and_p(d):
    p, _ = is_prime_power(d, get_data=True)
    n_d = (6 if p == 2 else 5 if p == 3 else 4) * d      # (8.3)
    return n_d, p

def admissible_orders(n, d, p):
    # Even orders o | 2n with d*gcd(2,p) | o, o > 2.  Only even orders are
    # needed: an alpha of odd order o is -alpha' for alpha' of order 2o, and
    # p^+_{alpha} = p^+_{-alpha}, so odd orders would only duplicate rows.
    D = d * gcd(2, p)
    return [o for o in divisors(2*n) if o > 2 and o % 2 == 0 and o % D == 0]

def weights_per_order(o):
    # Number of independent x^{2m} weights for the Galois orbit of alpha of
    # order o = number of distinct alpha^2 among conjugates:
    #   phi(o)/4 if 4 | o   (-alpha is a conjugate of alpha)
    #   phi(o)/2 otherwise  (-alpha has odd order o/2, not in the orbit)
    return ceil(euler_phi(o)/4) if o % 4 == 0 else ceil(euler_phi(o)/2)

# Sanity: per-block row count must reproduce the paper's m_{d,n} (8.2)
for d in D_LIST:
    n_d, p = nd_and_p(d)
    for n in range(d, n_d + 1, d):
        m_dn = ceil(sum(euler_phi(2*n // i)
                        for i in divisors(2*n // (d * gcd(2, p)))) / 4)
        ours = sum(weights_per_order(o) for o in admissible_orders(n, d, p))
        assert ours == m_dn, (d, n, ours, m_dn)
print("row counts match m_{d,n} for all d in", D_LIST)

# Max total degree ever fed to the Phi-reduction:
# x^{2m} * (trace of p^+_{alpha,n}) has total degree 2n + 2m  (kappa = 0).
# Compare the old f_n * g_{d,n} route: 4*n_d + deg(g) + 2m.
max_deg = 0
for d in D_LIST:
    n_d, p = nd_and_p(d)
    for n in range(d, n_d + 1, d):
        for o in admissible_orders(n, d, p):
            max_deg = max(max_deg, 2*n + 2*(weights_per_order(o) - 1))
print("max_deg =", max_deg)

In [ ]:
# ---------------- CELL 2: build the 3-index reduction cache ----------------
# Same recursion as before (sigma_z (5.1) and rho (5.2), kappa = 0, monomials
# kept sorted a >= b >= c), but now the z > 0 slices are STORED instead of
# discarded, keyed by (n/2, b/2, c/2).
import sqlite3, os, traceback, time

DB = "reduction_cache3.db"

try:
    if os.path.exists(DB):
        os.remove(DB)
    conn = sqlite3.connect(DB)
    cur = conn.cursor()
    cur.execute("PRAGMA journal_mode=WAL;")
    cur.execute("PRAGMA synchronous=NORMAL;")
    cur.execute("CREATE TABLE IF NOT EXISTS cache("
                " n_half INTEGER, b_half INTEGER, c_half INTEGER, poly TEXT,"
                " PRIMARY KEY (n_half, b_half, c_half))")
    conn.commit()

    R.<x, y, z> = PolynomialRing(ZZ)

    # Pruning: queried monomials have c0 = min exponent <= n0/3, and each
    # recursion step preserves or lowers n + c, so n + c <= (4/3) max_deg
    # covers everything reachable.
    def c_cap(n):
        return (4 * max_deg) // 3 - n

    temp_cache = {}
    cur_cache = {}

    def reduce_monomial(n, b, c):
        a = n - b - c                       # sorted monomial x^a y^b z^c
        if b not in cur_cache:
            cur_cache[b] = []
        if b == 0 and c == 0:
            cur_cache[0].append(x**a)
        elif c == 0:                        # sigma_z: x^a y^b -> 2 x^(a-1) y^(b-1) z
            t = sorted([a - 1, b - 1, 1], reverse=True)
            cur_cache[b].append(2 * temp_cache[t[1]][t[2]])
        else:                               # rho with kappa = 0
            t1 = sorted([a - 1, b + 1, c - 1], reverse=True)
            t2 = sorted([a - 1, b - 1, c + 1], reverse=True)
            cur_cache[b].append(temp_cache[b - 1][c - 1]
                                + temp_cache[t1[1]][t1[2]]
                                + temp_cache[t2[1]][t2[2]])

    t0 = time.time()
    counter = 0
    for n in range(max_deg + 1):
        for b in range(n // 2 + 1):
            for c in range(min(b, n - 2*b, c_cap(n)) + 1):
                reduce_monomial(n, b, c)
        if n % 2 == 0:
            for b in range(0, n // 2 + 1, 2):
                for c in range(0, min(b, n - 2*b, c_cap(n)) + 1, 2):
                    cur.execute("INSERT OR REPLACE INTO cache VALUES (?,?,?,?)",
                                (int(n // 2), int(b // 2), int(c // 2),
                                 str(cur_cache[b][c])))
                    counter += 1
            conn.commit()
        temp_cache = cur_cache
        cur_cache = {}
    conn.commit()
    temp_cache.clear()
    print("stored %d reductions up to degree %d in %.1fs"
          % (counter, max_deg, time.time() - t0))
except Exception:
    traceback.print_exc()

In [ ]:
# ---------------- CELL 3: eigenvector pipeline + verification ----------------
import sqlite3, time
from functools import lru_cache

DB = "reduction_cache3.db"
conn = sqlite3.connect(DB)
cur = conn.cursor()

RZ = PolynomialRing(ZZ, names=('x', 'y', 'z'))
xZ, yZ, zZ = RZ.gens()

@lru_cache(maxsize=int(200000))
def get_reduction(n_half, b_half, c_half):
    cur.execute("SELECT poly FROM cache WHERE n_half=? AND b_half=? AND c_half=?",
                (int(n_half), int(b_half), int(c_half)))
    row = cur.fetchone()
    if row is None:
        raise ValueError("cache missing " + str((n_half, b_half, c_half)))
    return RZ(row[0])

# ---- block matrices, (6.1)-(6.5) ----
def construct_A_block(i, ring):
    if i == 0:
        return Matrix(ring, 1, 1, [2])
    A = Matrix(ring, i + 1, i + 1, sparse=True)
    for k in range(i):
        A[k, k + 1] = 1
        A[k + 1, k] = 1
    A[1, 0] = 2        # (6.1): x*(first elt)  = 2*(second elt)
    A[i - 1, i] = 2    # (6.3): x*(last elt)   = 2*(penultimate) + B-part
    return A           # note: A_1 comes out [[0,2],[2,0]] automatically

def construct_B_block(i, ring):
    B = Matrix(ring, i, i + 1, sparse=True)
    for k in range(i):
        B[k, k + 1] = 1
    return B

# ---- p_{alpha,n} by block back-substitution (no kernels needed) ----
def eigenvector_blocks(n, o):
    # p_{alpha,n} for alpha = zeta_o + zeta_o^{-1}.  Top block is (6.6);
    # each lower block solves (A_i - alpha) v_i = -B_{i+1} v_{i+1}, which is
    # consistent even when singular (Thm 6.10).  Returns (K, alpha, [v_n..v_0]).
    K = CyclotomicField(o)
    zeta = K.gen()
    alpha = zeta + zeta**(-1)
    v_top = vector(K, [1] + [zeta**k + zeta**(-k) for k in range(1, n)]
                      + [zeta**n])
    A_n = construct_A_block(n, K)
    assert A_n * v_top == alpha * v_top, "(6.6) not an eigenvector of A_n"
    blocks = [v_top]
    for i in range(n - 1, -1, -1):
        rhs = -(construct_B_block(i + 1, K) * blocks[-1])
        S = construct_A_block(i, K) - alpha
        blocks.append(S.solve_right(rhs))   # raises if inconsistent
    assert blocks[-1][0] == 0, "omega != 0: not in the admissible eigenspace"
    return K, alpha, blocks

# ---- trace of p^+ : an integer polynomial with all exponents even ----
def traced_pplus_polynomial(n, o):
    K, alpha, blocks = eigenvector_blocks(n, o)
    RK = PolynomialRing(K, names=('x', 'y', 'z'))
    xK, yK, zK = RK.gens()
    poly = RK(0)
    for v_i, i in zip(blocks, range(n, -1, -1)):
        fac = (xK**2)**(n - i)              # kappa = 0: (x^2 - kappa)^(n-i)
        for k in range(i + 1):
            if k % 2:                       # p^+: even powers of y,z only
                continue
            poly += v_i[k] * fac * yK**(2*i - k) * zK**k
    tr = {m: QQ(c.trace()) for m, c in poly.dict().items()}
    tr = {m: v for m, v in tr.items() if v != 0}
    if not tr:
        return RZ(0)
    den = lcm([v.denominator() for v in tr.values()])
    return RZ({m: ZZ(den * v) for m, v in tr.items()})    # scaling is harmless

# ---- Phi via the cache ----
def reduce_int(content, n_d):
    if content == 0:
        return ZZ(0)
    content = ZZ(content)
    for q in prime_range(2 * n_d):
        content = content // q**valuation(content, q)
    return content

def reduce_polynomial_with_cache(poly, m, n_d):
    # Phi(x^{2m} * poly) as a polynomial in x (kappa = 0).
    out = RZ(0)
    for mono, coeff in zip(poly.monomials(), poly.coefficients()):
        e = list(mono.exponents()[0])
        assert e[0] % 2 == 0 and e[1] % 2 == 0 and e[2] % 2 == 0, \
            "odd exponent in " + str(mono)
        h = sorted([e[0]//2 + m, e[1]//2, e[2]//2], reverse=True)
        out += coeff * get_reduction(h[0] + h[1] + h[2], h[1], h[2])
    if out != 0:
        c = ZZ(out.content())
        if c > 1:
            out = reduce_int(c, n_d) * out // c
    return out

# ---- main verification ----
def verify_for_d(d):
    n_d, p = nd_and_p(d)
    t0 = time.time()
    rows = []
    for n in range(d, n_d + 1, d):
        n_rows = 0
        for o in admissible_orders(n, d, p):
            T = traced_pplus_polynomial(n, o)
            if T == 0:
                continue
            for m in range(weights_per_order(o)):
                red = reduce_polynomial_with_cache(T, m, n_d)
                # end-to-end check: Phi(x^{2m} Tr p^+) must collapse to deg <= 2n
                assert red == 0 or red.degree(xZ) <= 2*n, \
                    "degree %s > %s at (n=%s, o=%s, m=%s)" % (red.degree(xZ), 2*n, n, o, m)
                rows.append([ZZ(red.coefficient({xZ: i}))
                             for i in range(6, 2*n_d + 1, 2)])
                n_rows += 1
        print("  d=%s, n=%s: %s rows" % (d, n, n_rows))
    M = Matrix(ZZ, rows)
    need = n_d - 2
    print("d=%s: matrix %s x %s (need column rank %s), built in %.1fs"
          % (d, M.nrows(), M.ncols(), need, time.time() - t0))
    assert M.ncols() == need
    if M.nrows() < need:
        print("d=%s: FAIL (only %s rows)" % (d, M.nrows()))
        return False
    divs = M.elementary_divisors()          # prod = gcd of all maximal minors
    nz = [e for e in divs if e != 0]
    if len(nz) < M.ncols():
        print("d=%s: FAIL (rank %s < %s)" % (d, M.rank(), M.ncols()))
        return False
    g = reduce_int(prod(nz), n_d)
    print("d=%s: %s" % (d, "SUCCESS" if g == 1 else "FAIL (gcd = %s)" % g))
    return g == 1

def main():
    results = {}
    for d in D_LIST:
        results[d] = verify_for_d(d)        # inside the loop!
    print("Successful:", [d for d in results if results[d]])
    print("Failed:    ", [d for d in results if not results[d]])

main()